# Data download
Downloads three phishing email datasets from Kaggle, joins them, and saves three files to `Data/`:
- `phishing_combined.csv` – every email from all three datasets
- `phishing_general.csv` – everything except EduPhish (used for training)
- `phishing_education.csv` – EduPhish only

Run this first. Every other notebook reads the files it produces.
Kaggle credentials come from Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`).

In [20]:
# Connect this notebook to my Google Drive
# This lets the notebook save files into my Drive so they are not lost when Colab shuts down.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# Kaggle needs a username and a key before it lets me download anything.
# This cell reads them from there and hands them to Kaggle.

import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle credentials loaded.")

Kaggle credentials loaded.


In [22]:
# kagglehub downloads each dataset and tells me which folder it put it in.
# I store those folder names in path1, path2 and path3 so I can use them later.

import kagglehub

path1 = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")            # 6 general email files
path2 = kagglehub.dataset_download("tanvirahmed0981/education-targeted-phishing-email-dataset")  # Education emails (EduPhish)
path3 = kagglehub.dataset_download("kuladeep19/phishing-and-legitimate-emails-dataset")  # Synthetic dataset

print("Dataset 1 folder:", path1)
print("Dataset 2 folder:", path2)
print("Dataset 3 folder:", path3)

100%|██████████| 77.1M/77.1M [00:00<00:00, 116MB/s]

Extracting files...


Using Colab cache for faster access to the 'education-targeted-phishing-email-dataset' dataset.
Dataset 1 folder: /root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1
Dataset 2 folder: /kaggle/input/education-targeted-phishing-email-dataset
Dataset 3 folder: /root/.cache/kagglehub/datasets/kuladeep19/phishing-and-legitimate-emails-dataset/versions/1


In [23]:
# I want to have a look at what is inside each folder
# This prints every file in each dataset so I can see what I downloaded before I start using it.

import os

for name, folder in [("DATASET 1", path1), ("DATASET 2", path2), ("DATASET 3", path3)]:
    print("\n===", name, "===")
    for root, dirs, files in os.walk(folder):      # walk = look inside every sub-folder too
        for f in files:
            print(os.path.join(root, f))


=== DATASET 1 ===
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/CEAS_08.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/Nazario.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/SpamAssasin.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/Ling.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/Enron.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/Nigerian_Fraud.csv
/root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1/phishing_email.csv

=== DATASET 2 ===
/kaggle/input/education-targeted-phishing-email-dataset/EduPhish_Kaggle_Package/eduphish_dataset.csv
/kaggle/input/education-targeted-phishing-email-dataset/EduPhish_Kaggle_Package/DATASET_STRUCTURE.txt
/kaggle/input/education-targeted-phishing-email-dataset/EduPhish_Kaggle_Pa

In [25]:
# Load DATASET 1 (six separate email files)
# Each file is a spreadsheet of emails. For every file I:
#   - join the subject line and the email body into one piece of text
#   - keep the label (0 = legitimate, 1 = phishing)
#   - write down which file the email came from
# Then I stack all six into one big table called "dataset1".

import pandas as pd   # pandas is the tool used for working with tables

files1 = ["SpamAssasin.csv", "Enron.csv", "CEAS_08.csv",
          "Nazario.csv", "Nigerian_Fraud.csv", "Ling.csv"]

pieces = []                                   # an empty list to collect each file's table
for f in files1:
    d = pd.read_csv(os.path.join(path1, f))   # open the file as a table

    # Some files have a "subject" column, some do not. Use it if it exists.
    if "subject" in d.columns:
        subject = d["subject"].fillna("").astype(str)
    else:
        subject = ""
    body = d["body"].fillna("").astype(str)   # fillna("") turns empty cells into blank text

    text = (subject + " " + body).str.strip()          # join subject + body (the subject often shows phishing clues)
    label = pd.to_numeric(d["label"], errors="coerce")  # make sure the label is a number; anything odd becomes blank

    pieces.append(pd.DataFrame({"text": text,
                                "label": label,
                                "source": f[:-4]}))   # f[:-4] removes ".csv" from the file name

dataset1 = pd.concat(pieces, ignore_index=True)       # stack the six tables on top of each other
print("Dataset 1 rows:", len(dataset1))


Dataset 1 rows: 82486


In [26]:
# Now load DATASET 2 (EduPhish - education phishing emails)
# This dataset already has "text" and "label" columns, so I just keep those two and add a "source" column so I know where the emails came from.

csv2 = os.path.join(path2, "EduPhish_Kaggle_Package", "eduphish_dataset.csv")

dataset2 = pd.read_csv(csv2)[["text", "label"]].copy()
dataset2["source"] = "eduphish"
print("Dataset 2 rows:", len(dataset2))


Dataset 2 rows: 16942


In [27]:
# Load DATASET 3 (AI-generated phishing and legitimate emails).
# This dataset is one CSV file with five columns: text, label, phishing_type, severity and confidence. I only need "text" and "label".
# I do not know the exact file name, so this cell finds the first .csv in the folder.
# NOTE for the dissertation: these emails were written by an AI model, not real people.

csv3 = None
for root, dirs, files in os.walk(path3):
    for f in files:
        if f.endswith(".csv"):
            csv3 = os.path.join(root, f)
print("Using file:", csv3)

dataset3 = pd.read_csv(csv3)[["text", "label"]].copy()
dataset3["source"] = "kuladeep_synthetic"
print("Dataset 3 rows:", len(dataset3))


Using file: /root/.cache/kagglehub/datasets/kuladeep19/phishing-and-legitimate-emails-dataset/versions/1/phishing_legit_dataset_KD_10000.csv
Dataset 3 rows: 10000


In [28]:
# Clean up a table.
# All three tables need the same tidy-up, so I write the steps once as a "function" (a reusable recipe) and then use it three times.
# The steps:
#   - turn the text into plain text and remove spaces from the start and end
#   - make sure the label is a number
#   - throw away rows with no text or no label

def clean(table):
    table = table.copy()
    table["text"] = table["text"].fillna("").astype(str).str.strip()
    table["label"] = pd.to_numeric(table["label"], errors="coerce")
    table = table[(table["text"].str.len() > 0) & table["label"].notna()]   # keep only good rows
    table["label"] = table["label"].astype(int)                              # label as a whole number (0 or 1)
    return table.reset_index(drop=True)                                      # renumber the rows 0, 1, 2 ...

dataset1 = clean(dataset1)
dataset2 = clean(dataset2)
dataset3 = clean(dataset3)


In [29]:
# Build FILE 1 - all three datasets combined.
# Stack the three tables into one, then remove any email that appears more than once (duplicates would let the model "cheat" by seeing the same email twice).

combined = pd.concat([dataset1, dataset2, dataset3], ignore_index=True)

before = len(combined)
combined = combined.drop_duplicates(subset="text").reset_index(drop=True)   # keep the first copy of each email
print("Removed", before - len(combined), "duplicate emails")

print("\nCOMBINED - total rows:", len(combined))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(combined["label"].value_counts())
print("\nRows per source:")
print(combined["source"].value_counts())

combined.to_csv("phishing_combined.csv", index=False)
print("\nSaved -> phishing_combined.csv")

Removed 4075 duplicate emails

COMBINED - total rows: 105353

Label balance (0 = legitimate, 1 = phishing):
label
1    54857
0    50496
Name: count, dtype: int64

Rows per source:
source
CEAS_08               39145
Enron                 29745
eduphish              12956
kuladeep_synthetic     9956
SpamAssasin            5809
Nigerian_Fraud         3319
Ling                   2859
Nazario                1564
Name: count, dtype: int64

Saved -> phishing_combined.csv


In [19]:
# Build FILE 2 - education emails only.
# This is just the EduPhish dataset on its own (both phishing and legitimate emails), so I can test how well a model does on education-specific emails.
# If I ever want ONLY the phishing ones, I would add this line:
#     education = education[education["label"] == 1]

education = dataset2.drop_duplicates(subset="text").reset_index(drop=True)

print("EDUCATION - total rows:", len(education))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(education["label"].value_counts())

education.to_csv("phishing_education.csv", index=False)
print("\nSaved -> phishing_education.csv")

EDUCATION - total rows: 16942

Label balance (0 = legitimate, 1 = phishing):
label
0    9125
1    7817
Name: count, dtype: int64

Saved -> phishing_education.csv


In [24]:
# I need to read the licence and notes for the EduPhish dataset
# The EduPhish dataset comes with a licence file. I need to read it and mention it in the ethics section of my dissertation.

pkg = os.path.join(path2, "EduPhish_Kaggle_Package")

print("--- DATASET_STRUCTURE.txt ---")
print(open(os.path.join(pkg, "DATASET_STRUCTURE.txt")).read())

print("\n--- LICENSE_NOTICE.txt ---")
print(open(os.path.join(pkg, "LICENSE_NOTICE.txt")).read())

--- DATASET_STRUCTURE.txt ---
eduphish_dataset.csv

Columns recommended:
- text
- label


--- LICENSE_NOTICE.txt ---
GNU LESSER GENERAL PUBLIC LICENSE
Version 3, 29 June 2007

This dataset (EduPhish) is a derivative compilation of publicly available datasets that include components licensed under the GNU Lesser General Public License (LGPL-3.0) and other open licenses.

In accordance with upstream licensing requirements:

This derivative dataset is distributed under LGPL-3.0.

Redistribution and modification are permitted under the terms of LGPL-3.0.

A copy of the full LGPL-3.0 license text must accompany this distribution.

Original copyright and license notices of source datasets must be preserved.

Upstream datasets remain under their respective licenses.
Users are responsible for complying with the terms of each original dataset.

The authors of EduPhish do not claim ownership of the original email texts and provide this dataset for research purposes only.


In [31]:
# Build FILE 3 - general (non-education) emails only.
# This is everything EXCEPT EduPhish: the six Dataset 1 sources plus the synthetic dataset. I will train models on this file and test them on the education file, to see whether a model built on general emails still works on education-targeted ones.

general = combined[combined["source"] != "eduphish"].reset_index(drop=True)

print("GENERAL - total rows:", len(general))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(general["label"].value_counts())
print("\nRows per source:")
print(general["source"].value_counts())

general.to_csv("phishing_general.csv", index=False)
print("\nSaved -> phishing_general.csv")

GENERAL - total rows: 92397

Label balance (0 = legitimate, 1 = phishing):
label
1    48846
0    43551
Name: count, dtype: int64

Rows per source:
source
CEAS_08               39145
Enron                 29745
kuladeep_synthetic     9956
SpamAssasin            5809
Nigerian_Fraud         3319
Ling                   2859
Nazario                1564
Name: count, dtype: int64

Saved -> phishing_general.csv


In [ ]:
# Keep a copy of the EduPhish licence next to the data.
# This is for my records and for the ethics section.

import shutil
shutil.copy(os.path.join(path2, "EduPhish_Kaggle_Package", "LICENSE_NOTICE.txt"),
            "EduPhish_LICENSE_NOTICE.txt")

'EduPhish_LICENSE_NOTICE.txt'

In [34]:
# Copy the finished files into my "Master's project" folder in Drive
# Colab deletes its own files when it shuts down, so I must copy them to Drive to keep them. This cell looks for a folder whose name starts with "Master" inside "Colab Notebooks" and copies the files there.

notebooks = "/content/drive/MyDrive/Colab Notebooks"

folder = None
for name in os.listdir(notebooks):
    if name.lower().startswith("master") and os.path.isdir(os.path.join(notebooks, name)):
        folder = name

if folder is None:
    print("Could not find a folder starting with 'Master' in Colab Notebooks. Folders found:")
    for name in sorted(os.listdir(notebooks)):
        if os.path.isdir(os.path.join(notebooks, name)):
            print("  ", repr(name))
else:
    dest = os.path.join(notebooks, folder, "Data")   # the data folder inside Master's project
    os.makedirs(dest, exist_ok=True)                 # create it if it does not exist yet
    shutil.copy("phishing_combined.csv", dest)
    shutil.copy("phishing_education.csv", dest)
    shutil.copy("phishing_general.csv", dest)
    shutil.copy("EduPhish_LICENSE_NOTICE.txt", dest)
    print("Copied into:", repr(dest))
    print("Folder now contains:", os.listdir(dest))
    print("Folder now contains:", os.listdir(dest))

Copied into: "/content/drive/MyDrive/Colab Notebooks/Master's project/data"
Folder now contains: ['phishing_combined.csv', 'phishing_education.csv', 'phishing_general.csv', 'EduPhish_LICENSE_NOTICE.txt']
Folder now contains: ['phishing_combined.csv', 'phishing_education.csv', 'phishing_general.csv', 'EduPhish_LICENSE_NOTICE.txt']


In [35]:
# Check the saved files by reading them back from Drive.
# If this cell prints sensible numbers, everything worked.

drive_folder = "/content/drive/MyDrive/Colab Notebooks/Master's project/Data"

combined = pd.read_csv(os.path.join(drive_folder, "phishing_combined.csv"))
education = pd.read_csv(os.path.join(drive_folder, "phishing_education.csv"))

print("=== COMBINED ===")
print("Total rows:", len(combined))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(combined["label"].value_counts())
print("\nAs percentages:")
print((combined["label"].value_counts(normalize=True) * 100).round(1))
print("\nRows per source:")
print(combined["source"].value_counts())

print("\n=== EDUCATION ONLY ===")
print("Total rows:", len(education))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(education["label"].value_counts())
print("\nAs percentages:")
print((education["label"].value_counts(normalize=True) * 100).round(1))

general = pd.read_csv(os.path.join(drive_folder, "phishing_general.csv"))
print("\n=== GENERAL ONLY ===")
print("Total rows:", len(general))
print("\nLabel balance (0 = legitimate, 1 = phishing):")
print(general["label"].value_counts())

=== COMBINED ===
Total rows: 105353

Label balance (0 = legitimate, 1 = phishing):
label
1    54857
0    50496
Name: count, dtype: int64

As percentages:
label
1    52.1
0    47.9
Name: proportion, dtype: float64

Rows per source:
source
CEAS_08               39145
Enron                 29745
eduphish              12956
kuladeep_synthetic     9956
SpamAssasin            5809
Nigerian_Fraud         3319
Ling                   2859
Nazario                1564
Name: count, dtype: int64

=== EDUCATION ONLY ===
Total rows: 16942

Label balance (0 = legitimate, 1 = phishing):
label
0    9125
1    7817
Name: count, dtype: int64

As percentages:
label
0    53.9
1    46.1
Name: proportion, dtype: float64

=== GENERAL ONLY ===
Total rows: 92397

Label balance (0 = legitimate, 1 = phishing):
label
1    48846
0    43551
Name: count, dtype: int64


In [38]:
#Liscence notice for edudata because of issues with data_clean
import kagglehub, os
path2 = kagglehub.dataset_download("tanvirahmed0981/education-targeted-phishing-email-dataset")

pkg = os.path.join(path2, "EduPhish_Kaggle_Package")
for f in sorted(os.listdir(pkg)):
    print(f)

print("\n--- DATASET_STRUCTURE.txt ---")
print(open(os.path.join(pkg, "DATASET_STRUCTURE.txt")).read())

print("\n--- LICENSE_NOTICE.txt ---")
print(open(os.path.join(pkg, "LICENSE_NOTICE.txt")).read())

print(open(os.path.join(pkg, "README.md")).read())

Using Colab cache for faster access to the 'education-targeted-phishing-email-dataset' dataset.
DATASET_STRUCTURE.txt
LICENSE_NOTICE.txt
README.md
eduphish_dataset.csv

--- DATASET_STRUCTURE.txt ---
eduphish_dataset.csv

Columns recommended:
- text
- label


--- LICENSE_NOTICE.txt ---
GNU LESSER GENERAL PUBLIC LICENSE
Version 3, 29 June 2007

This dataset (EduPhish) is a derivative compilation of publicly available datasets that include components licensed under the GNU Lesser General Public License (LGPL-3.0) and other open licenses.

In accordance with upstream licensing requirements:

This derivative dataset is distributed under LGPL-3.0.

Redistribution and modification are permitted under the terms of LGPL-3.0.

A copy of the full LGPL-3.0 license text must accompany this distribution.

Original copyright and license notices of source datasets must be preserved.

Upstream datasets remain under their respective licenses.
Users are responsible for complying with the terms of each or